# RoboCasa Kitchen Benchmark: RLDX-1 on `TurnOnSinkFaucet`

RoboCasa is **not** a `physicalai-train` extra: its `setup.py` pins
`lerobot==0.3.3` (conflicts with this repo's `lerobot>=0.5.1`) and it needs
robosuite's `master` branch (conflicts with the `[libero]` extra's pinned
`robosuite==1.4.0`). It must live in its own venv and its own **Jupyter
kernel** -- unlike `policies/rldx1.ipynb`, this notebook cannot share a
kernel with the rest of the library.

### How to run this notebook

1. Run the **Setup** cell below (creates `library/.venv-robocasa`, installs
   robocasa + robosuite, downloads Kitchen assets). This can be run from any
   kernel since it only shells out.
2. **Switch this notebook's kernel** to `library/.venv-robocasa/bin/python`
   (Command Palette -> "Jupyter: Select Kernel" -> "Select Another Kernel" ->
   "Existing Python Environment" -> browse to that interpreter).
3. Run the remaining cells normally -- from here on it's regular inline
   `physicalai` code, no subprocess/external script needed.

---
## 1. Setup: create the RoboCasa venv and download Kitchen assets

Creates `library/.venv-robocasa` (skips creation if it already exists), then
installs robocasa + robosuite via `library/scripts/benchmark/install_robocasa.sh`.

In [ ]:
%%bash
set -euo pipefail
ROOT="$(git rev-parse --show-toplevel)"
cd "$ROOT/library"

if [ ! -d .venv-robocasa ]; then
    uv venv .venv-robocasa
fi
source .venv-robocasa/bin/activate
# Pick exactly one torch backend extra matching your hardware.
uv sync --active --extra cu128
bash scripts/benchmark/install_robocasa.sh

Now download the Kitchen assets. 

In [ ]:
%%bash
set -euo pipefail
ROOT="$(git rev-parse --show-toplevel)"
cd "$ROOT/library"
source .venv-robocasa/bin/activate
export MUJOCO_GL=egl

yes y | python -m robocasa.scripts.download_kitchen_assets --type tex tex_generative fixtures_lw objs_lw objs_objaverse

---
## ⚠️ Switch kernel now

Before running any cell below, switch this notebook's kernel to
`library/.venv-robocasa/bin/python`
(Command Palette -> **Jupyter: Select Kernel** -> **Select Another Kernel** ->
**Existing Python Environment** -> browse to `library/.venv-robocasa/bin/python`).

The cell below just confirms the switch worked.

In [ ]:
import sys

import robocasa  # noqa: F401
import robosuite  # noqa: F401

assert ".venv-robocasa" in sys.executable, (
    f"Wrong kernel: {sys.executable}. Switch to library/.venv-robocasa/bin/python."
)
print(f"Using interpreter: {sys.executable}")
print("robocasa + robosuite import OK")

---
## 2. Load the RLDX-1-FT-ROBOCASA policy

`use_bf16`/`backbone_trainable_params_fp32`/`tune_top_llm_layers` match the
checkpoint's own training config.

In [ ]:
# Copyright (C) 2026 Intel Corporation
# SPDX-License-Identifier: Apache-2.0

import subprocess
from pathlib import Path

import torch

from physicalai.policies.rldx1 import Rldx1

REPO_ROOT = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())

policy = Rldx1(
    pretrained_name_or_path="RLWRLD/RLDX-1-FT-ROBOCASA",
    use_bf16=True,  # matches the checkpoint's single-precision bf16 weights
    backbone_trainable_params_fp32=True,
    tune_top_llm_layers=4,
    attn_implementation="flash_attention_2",
)
policy.eval()
policy.to("cuda")

---
## 3. Benchmark: `TurnOnSinkFaucet`


In [ ]:
from physicalai.benchmark.gyms import Benchmark
from physicalai.gyms import RoboCasaGym

TASK = "TurnOnSinkFaucet"
NUM_EPISODES = 5
robocasa_video_dir = REPO_ROOT / "robocasa_turn_on_sink_faucet_videos"

gym = RoboCasaGym(task=TASK)

benchmark = Benchmark(
    gyms=[gym],
    num_episodes=NUM_EPISODES,
    video_dir=robocasa_video_dir,
    record_mode="all",
)
# Stack the 3 default camera views horizontally, matching RoboCasaBenchmark's convention.
benchmark.frame_key = ["robot0_agentview_left", "robot0_agentview_right", "robot0_eye_in_hand"]

with torch.autocast("cuda", dtype=torch.bfloat16):
    robocasa_results = benchmark.evaluate(policy)

print(robocasa_results.summary())

In [ ]:
from IPython.display import Video, display

robocasa_video_paths = sorted(robocasa_video_dir.glob("*.mp4"))
print(f"Found {len(robocasa_video_paths)} RoboCasa episode videos in {robocasa_video_dir}")

for video_path in robocasa_video_paths:
    print(video_path.name)
    display(Video(str(video_path), embed=False, width=640))